# pyomo elements
![](Images/2022-11-26-13-43-05.png)




## There are two types of models
###  1- Abstract models
 ![](Images/2022-11-26-13-44-19.png)
### 2- Concrete models
 ![](Images/2022-11-26-13-44-37.png)

 ![](Images/2022-11-26-13-45-43.png)

In [1]:
# import pyomo (Ahmed)
import os
import sys
from pathlib import Path
from pyomo.environ import *

# Windows conda: put Library/bin on PATH so Pyomo can find glpsol
_glpk_bin = Path(sys.prefix) / "Library" / "bin"
if _glpk_bin.exists():
    os.environ["PATH"] = str(_glpk_bin) + os.pathsep + os.environ.get("PATH", "")

![](Images/2022-11-25-15-33-48.png)

In [2]:
# Modelling the problem

# Intialize the model
model = AbstractModel()

# Define the variables
model.x1 = Var(bounds=(0,None), within= NonNegativeReals)
model.x2 = Var(bounds=(0,None), within= NonNegativeReals)

# Define the constrains
model.eq1 = Constraint(expr=2*model.x1 + 3*model.x2<=4)
model.eq2 = Constraint(expr=9*model.x1 + 1*model.x2<=6)

# Define the objective function
model.obj = Objective(expr=3* model.x1 + model.x2 , sense=maximize)

In [3]:
from pyomo.environ import SolverFactory
print(SolverFactory('glpk').available(exception_flag=False))
print(SolverFactory('glpk').executable())

True
c:\Users\Ahmed_Awaad\.conda\envs\main\Library\bin\glpsol.exe


In [4]:
# Solving the problem

# Initialize the solver   --> glpk open source for linear programming
opt = SolverFactory('glpk')

# create the instance for concrete problem  (in this problem we will not use file for instance parameters because we defined it in the model)
instance = model.create_instance()

# Get the results  ((Solve and update the instance))
results = opt.solve(instance)

# check if solution is feasible or not
if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
    print('feasible')
elif (results.solver.termination_condition ==TerminationCondition.infeasible):
    print('infeasible')
else:
    print('solver status: ', results.solver.status)


feasible


In [5]:
results

{'Problem': [{'Name': 'unknown', 'Lower bound': 2.64, 'Upper bound': 2.64, 'Number of objectives': 1, 'Number of constraints': 2, 'Number of variables': 2, 'Number of nonzeros': 4, 'Sense': 'maximize'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Statistics': {'Branch and bound': {'Number of bounded subproblems': 0, 'Number of created subproblems': 0}}, 'Error rc': 0, 'Time': 0.049179792404174805}], 'Solution': [OrderedDict([('number of solutions', 0), ('number of solutions displayed', 0)])]}

In [6]:
# check for instance equations
instance.eq1.pprint()
instance.eq2.pprint()

eq1 : Size=1, Index=None, Active=True
    Key  : Lower : Body        : Upper : Active
    None :  -Inf : 2*x1 + 3*x2 :   4.0 :   True
eq2 : Size=1, Index=None, Active=True
    Key  : Lower : Body      : Upper : Active
    None :  -Inf : 9*x1 + x2 :   6.0 :   True


In [7]:
# print the optimal values
print('X1= ', value(instance.x1))
print('X2= ', value(instance.x2))
print('OF= ', value(instance.obj))

X1=  0.56
X2=  0.96
OF=  2.64


# Update the parameter values

In [8]:
##########################################################################################################################
# Modelling the problem
##########################################################################################################################
# Initialize the model
model = AbstractModel()

# Define the variables
model.x1 = Var(bounds=(0,None), within= NonNegativeReals)
model.x2 = Var(bounds=(0,None), within= NonNegativeReals)

"""Define the parameters"""
model.a11 = Param(mutable=True)

# Define the constrains
model.eq1 = Constraint(expr=model.a11*model.x1 + 3*model.x2<=4)
model.eq2 = Constraint(expr=9*model.x1 + 1*model.x2<=6)

# Define the objective function
model.obj = Objective(expr=3* model.x1 + model.x2 , sense=maximize)

##########################################################################################################################
# Solving the problem
##########################################################################################################################
# Initialize the solver   --> glpk open source for linear programming
opt = SolverFactory('glpk')

# create the instance for concrete problem  (in this problem we will not use file for instance parameters because we defined it in the model)
instance = model.create_instance()

"""substiture the parameters values"""
instance.a11 = 7

# Get the results  ((Solve and update the instance))
results = opt.solve(instance)

# check if solution is feasible or not
if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
    print('feasible')
elif (results.solver.termination_condition ==TerminationCondition.infeasible):
    print('infeasible')
else:
    print('solver status: ', results.solver.status)

##########################################################################################################################
# Print the optimal values
##########################################################################################################################
print('________________________')
print('X1= ', value(instance.x1))
print('X2= ', value(instance.x2))
print('OF= ', value(instance.obj))

feasible
________________________
X1=  0.571428571428571
X2=  0.0
OF=  1.7142857142857129


# Another problem
![](Images/2022-11-25-16-26-01.png)

In [9]:
# Modelling the problem
model= AbstractModel()
model.x1 = Var(bounds=(0, None), within=NonNegativeReals)
model.x2 = Var(bounds=(0, None), within=NonNegativeReals)
model.con1 = Constraint(expr=model.x1+2* model.x2 <=8)
model.con2 = Constraint(expr=4*model.x1 - model.x2 <=2)
## Another way to define the constraints or objective function
def obj_rule(model):
    return 2* model.x1 +3* model.x2
model.OF = Objective(rule=obj_rule, sense=maximize)

# Solve the model
opt = SolverFactory('glpk')
instance = model.create_instance()
results = opt.solve(instance)

# check the results
if (results.solver.status== SolverStatus.ok) and (results.solver.termination_condition== TerminationCondition.optimal):
    print('Feasible')
elif (results.solver.termination_condition== TerminationCondition.infeaible):
    print('Infeasible')
else:
    print('Solver Status:' , results.solver.status)
    
# print the optimal results
print('X1: ', value(instance.x1))
print('X2: ', value(instance.x2))
print('OF: ', value(instance.OF))

Feasible
X1:  1.33333333333333
X2:  3.33333333333333
OF:  12.66666666666665


In [10]:
# analyze the results
results.write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Name: unknown
  Lower bound: 12.6666666666667
  Upper bound: 12.6666666666667
  Number of objectives: 1
  Number of constraints: 2
  Number of variables: 2
  Number of nonzeros: 4
  Sense: maximize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 0
      Number of created subproblems: 0
  Error rc: 0
  Time: 0.04924607276916504
# ----------------------------------------------------------
#   Solution Information
# ---------------

In [11]:
# analyze the problem
instance.pprint()

2 Var Declarations
    x1 : Size=1, Index=None
        Key  : Lower : Value            : Upper : Fixed : Stale : Domain
        None :     0 : 1.33333333333333 :  None : False : False : NonNegativeReals
    x2 : Size=1, Index=None
        Key  : Lower : Value            : Upper : Fixed : Stale : Domain
        None :     0 : 3.33333333333333 :  None : False : False : NonNegativeReals

1 Objective Declarations
    OF : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 2*x1 + 3*x2

2 Constraint Declarations
    con1 : Size=1, Index=None, Active=True
        Key  : Lower : Body      : Upper : Active
        None :  -Inf : x1 + 2*x2 :   8.0 :   True
    con2 : Size=1, Index=None, Active=True
        Key  : Lower : Body      : Upper : Active
        None :  -Inf : 4*x1 - x2 :   2.0 :   True

5 Declarations: x1 x2 con1 con2 OF


## Create a Dat file for AbstractModel
<br> You need to have a .dat file in your directory containing the data required for filling the instance 
<br> for example :
<br> 
### Ex1: You do have a parameter called cap in your model and this parameter is defined over two sets called i,j where
<br> i belongs to {0,1,2,3,4,5,6}
<br> j belongs to {0,1,2,3,4,5,6}
<br> 
> model.cap = Param(model.i,model.j, within=Reals, mutable=True)

<br> You just need to create a file with .dat extension and indicate the value for each combination of i and j (as a table) like this :
<br> 
<br> param cap      :  0 1  2 3 4 5 6  := 
<br>                 0 0 3  3 4 0 0 0
<br>                 1 0 0  0 0 2 0 0
<br>                 2 0 10 0 0 1 0 0 
<br>                 3 0 0  0 0 0 5 0 
<br>                 4 0 0  0 0 0 1 2
<br>                 5 0 0  0 0 0 0 5
<br>                 6 0 0  0 0 0 0 0;
<br> This will feed the cap values as well as the sets i,j to the model
<br>
### Ex2: You do have a parameter called N in your model and this parameter is a constant number and has no set
> model.N = Param(mutable=True)

<br> You just need to create a file with .dat extension and indicate the value of N like this :
<br> 
<br> param N:=8;
<br>
### Ex3: You do have a parameter called R over he set i in your model
<br>
<br> i belongs to {0,1,2,3,4,5,6}

> model.R = Param(model.i)

<br> You just need to create a file with .dat extension and indicate the value of R for each element of i  like this :
<br> 
<br> param: i: R:=
<br>         1  2
<br>         2  1.2
<br>         3  1.8
<br>         4  0.9
<br>         5  3.2
<br>         6  0.7;
<br> This will feed the R values as well as the set i to the model

## How to call a .dat file for initializing the instance in Abstract Models
<br>Instance can be initialized with a dat file
<br>First create your dat file as described and then save it as a Filename.dat
<br>Call your .dat file
> instance = model.create_instance("Filename.dat")

### Example

In [12]:
# create the dat file
a11_file = """
param a11:= 7;
"""
with open('01_a11.dat', 'w') as file:
    file.write(a11_file)

In [13]:
##########################################################################################################################
# Modelling the problem
##########################################################################################################################
# Initialize the model
model = AbstractModel()

# Define the variables
model.x1 = Var(bounds=(0,None), within= NonNegativeReals)
model.x2 = Var(bounds=(0,None), within= NonNegativeReals)

"""Define the parameters"""
model.a11 = Param(mutable=True)

# Define the constrains
model.eq1 = Constraint(expr=model.a11*model.x1 + 3*model.x2<=4)
model.eq2 = Constraint(expr=9*model.x1 + 1*model.x2<=6)

# Define the objective function
model.obj = Objective(expr=3* model.x1 + model.x2 , sense=maximize)

##########################################################################################################################
# Solving the problem
##########################################################################################################################
# Initialize the solver   --> glpk open source for linear programming
opt = SolverFactory('glpk')

# create the instance for concrete problem
# read the parameters values from the Dat file
instance = model.create_instance("a11.dat")

# Get the results  ((Solve and update the instance))
results = opt.solve(instance)

# check if solution is feasible or not
if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
    print('feasible')
elif (results.solver.termination_condition ==TerminationCondition.infeasible):
    print('infeasible')
else:
    print('solver status: ', results.solver.status)

##########################################################################################################################
# Print the optimal values
##########################################################################################################################
print('________________________')
print('X1= ', value(instance.x1))
print('X2= ', value(instance.x2))
print('OF= ', value(instance.obj))

OSError: Cannot find file 'a11.dat'